# Downloading gnomAD and ClinVar data

In [14]:
import os
import subprocess
import polars as pl
import pandas as pd
import re
# !conda install conda-forge::awscli

# Set strict error handling
def run_command(cmd):
    result = subprocess.run(cmd, shell=True, check=True, text=True)
    return result

# Define paths for gnomad
VCF_DIR = "raw_inputs/gnomad"
CHROMOSOMES = [str(c) for c in list(range(1, 23))] + ['X', 'Y']

## gnomAD exomes v4.1 download

In [13]:
# Define paths for gnomad
VCF_DIR = "raw_inputs/gnomad"
CHROMOSOMES = [str(c) for c in list(range(1, 23))] + ['X', 'Y']
aws = "/home/shenrunx/software/anaconda3/bin/aws" ## replace this with the path of aws on your machine if needed

# Create directories
os.makedirs(VCF_DIR, exist_ok=True)

# Check missing VCF files
missing_vcf = []
for chrom in CHROMOSOMES:
    vcf_file = f"{VCF_DIR}/gnomad.exomes.v4.1.sites.chr{chrom}.vcf.bgz"
    if not os.path.exists(vcf_file):
        missing_vcf.append(chrom)

# Download missing files
if not missing_vcf:
    print("All VCF files present. Skipping download.")
else:
    print(f"Downloading missing chromosomes: {', '.join(missing_vcf)}")
    for chrom in missing_vcf:
        print(f"Downloading chr{chrom}...")
        run_command(f"{aws} s3 cp --no-sign-request s3://gnomad-public-us-east-1/release/4.1/vcf/exomes/gnomad.exomes.v4.1.sites.chr{chrom}.vcf.bgz {VCF_DIR}/gnomad.exomes.v4.1.sites.chr{chrom}.vcf.bgz")

All VCF files present. Skipping download.


## ClinVar Annotations download

ClinVar variants released on 2024-03-01 and the data is accessed on March 13, 2025

In [ ]:
# Download with streaming (unchanged)
dest_path="raw_inputs/clinvar/variant_summary.txt.gz"
# !wget https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz -O ${dest_path}



### Clean up ClinVar annotations

For merging the ClinVar with the VarChAMP variants more easily

In [ ]:
# processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"

# # Read CSV with Polars (auto-decompresses .gz)
# clinvar = pl.read_csv(dest_path, separator='\t', infer_schema_length=1000000)
# ## Process data with Polars expressions
# clinvar = (
#     clinvar
#     .filter(pl.col("Assembly") == "GRCh38")
#     .with_columns(
#         pl.col("Name")
#         # Split the string on '.', '(' or ')'
#         .map_elements(lambda s: re.split(r"[.\(\)]", s), pl.List(pl.String))
#         .alias("Name_listed")
#     )
# )
# clinvar = clinvar.with_columns(
#     pl.col("Name_listed")
#     .list.to_struct(
#             fields=[
#             'NM', 'version', 'gene', 
#             'c', 'clinvar_nt_change', 
#             'p', 'clinvar_aa_change'
#             ],
#             upper_bound=7)
#     .struct.unnest()
# ).with_columns(
#         SPDI=pl.col("ChromosomeAccession") + ":" + 
#             pl.col("PositionVCF").cast(str) + ":" +
#             pl.col("ReferenceAlleleVCF") + ":" +
#             pl.col("AlternateAlleleVCF"),
#         RefSeq_mRNA=pl.col("NM") + "." + pl.col("version"),
#         StarStatus=pl.when(pl.col("ReviewStatus").str.contains(r'single|conflicting'))
#             .then(1)
#             .when(pl.col("ReviewStatus").str.contains('multiple'))
#             .then(2)
#             .when(pl.col("ReviewStatus").str.contains('expert'))
#             .then(3)
#             .when(pl.col("ReviewStatus").str.contains('practice'))
#             .then(4)
#             .otherwise(None)
# )

# ## Make sure we only drop columns that actually exist
# columns_to_drop = [col for col in ['NM', 'version', 'gene', 'c', 'p', 'ReferenceAllele', 'AlternateAllele', 'Name_listed'] 
#                   if col in clinvar.columns]
# if columns_to_drop:
#     clinvar = clinvar.drop(columns_to_drop)

# # Now cast to categorical
# clinvar = clinvar.with_columns(
#     pl.col(["Assembly", "ClinicalSignificance", "ReviewStatus"]).cast(pl.Categorical)
# )

# # Write to Parquet with ZSTD compression
# clinvar.write_parquet(processed_clinvar_parq, 
#                      compression="zstd", 
#                      statistics=True)

## Merge ClinVar and gnomad to varchamp alleles

### Loading varchamp alleles

In [3]:
meta_inputs = "../1_inputs"
# get unique info for genes
gene_info = pl.scan_csv(f"{meta_inputs}/VarChamp_Consolidated_allele_collection.tsv", separator="\t", infer_schema_length=100000) #[gene_cols]
gene_info.collect()

orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,mutation_id_rc4,mutation_id_cegs2,mutation_id_edgotyping3,entry_plate,entry_well,collection,sequence
i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str
2,"""GBA""","""160G>C""","""Val54Leu""","""ENSG00000177628""","""1""",155240033,"""NC_000001.11:155240033:C:G""","""ALE0000584""","""CCSBVarC000001""","""6""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""F12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…"
2,"""GBA""","""670C>T""","""Leu224Phe""","""ENSG00000177628""","""1""",155238225,"""NC_000001.11:155238225:G:A""","""ALE00000002""","""CCSBVarC000002""","""73""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""C12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…"
2,"""GBA""","""887G>A""","""Arg296Gln""","""ENSG00000177628""","""1""",155237453,"""NC_000001.11:155237453:C:T""","""ALE00000003""","""CCSBVarC000003""","""113""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""D12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…"
2,"""GBA""","""1448T>C""","""Leu483Pro""","""ENSG00000177628""","""1""",155235252,"""NC_000001.11:155235252:A:G""","""ALE00000004""","""CCSBVarC000004""","""231""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""E12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…"
25,"""SLC9A3R1""","""328C>G""","""Leu110Val""","""ENSG00000109062""","""17""",74749174,"""NC_000017.11:74749174:C:G""","""ALE00000005""","""CCSBVarC000005""","""252""","""200008""","""NULL""","""GDEhDisVCh_40045""","""E04""","""RC4""","""ATGAGCGCGGACGCAGCGGCCGGGGCGCCC…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
8999,"""TTR""","""355G>A""","""Asp119Asn""","""ENSG00000118271""","""18""",31598586,"""NC_000018.10:31598586:G:A""","""ALE00008468""","""CCSBVarC008481""","""NULL""","""NULL""","""390""","""NULL""","""NULL""","""Edgotyping3""","""ATGGCTTCTCATCGTCTGCTCCTCCTCTGC…"
7323,"""PMP22""","""152A>G""","""His51Arg""","""ENSG00000109099""","""17""",15259120,"""NC_000017.11:15259120:T:C""","""ALE00008469""","""CCSBVarC008466""","""NULL""","""NULL""","""394""","""NULL""","""NULL""","""Edgotyping3""","""ATGCTCCTCCTGTTGCTGAGTATCATCGTC…"
2651,"""CLN6""","""89G>C""","""Gly30Ala""","""ENSG00000128973""","""15""",68218645,"""NC_000015.10:68218645:C:G""","""ALE00008470""","""CCSBVarC008436""","""NULL""","""NULL""","""436""","""NULL""","""NULL""","""Edgotyping3""","""ATGGAGGCGACGCGGAGGCGGCAGCACCTG…"


### Merge with ClinVar

In [5]:
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"
gene_info_clinvar_parq = "../3_outputs/varchamp_consolidated_alleles_clinvar_annot.parquet"
gene_info_clinvar_tsv = "../3_outputs/varchamp_consolidated_alleles_clinvar_annot.tsv"

# clinvar = pl.scan_parquet(processed_clinvar_parq)
# clinvar.collect().head()
# gene_info_clinvar = gene_info.join(
#                         clinvar,
#                         left_on="spdi",
#                         right_on="SPDI",
#                         how="left"
#                     )
# gene_info_clinvar = gene_info_clinvar.collect()

# gene_info_clinvar.write_parquet(gene_info_clinvar_parq, 
#                                 compression="zstd")
# gene_info_clinvar.write_csv(gene_info_clinvar_tsv, separator="\t")

#### Report the number of alleles missing ClinVar annotations

In [7]:
gene_info_clinvar = pl.scan_parquet(gene_info_clinvar_parq)
gene_info_clinvar.filter((~pl.col("spdi").is_null())&(pl.col("#AlleleID").is_null())).collect()

orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,mutation_id_rc4,mutation_id_cegs2,mutation_id_edgotyping3,entry_plate,entry_well,collection,sequence,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus
i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32
2,"""GBA""","""160G>C""","""Val54Leu""","""ENSG00000177628""","""1""",155240033,"""NC_000001.11:155240033:C:G""","""ALE0000584""","""CCSBVarC000001""","""6""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""F12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""GBA""","""670C>T""","""Leu224Phe""","""ENSG00000177628""","""1""",155238225,"""NC_000001.11:155238225:G:A""","""ALE00000002""","""CCSBVarC000002""","""73""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""C12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
167,"""ACTA2""","""874A>G""","""Arg292Gly""","""ENSG00000107796""","""10""",88938177,"""NC_000010.11:88938177:T:C""","""ALE00000012""","""CCSBVarC000012""","""390""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""B12""","""RC4""","""ATGTGTGAAGAAGAGGACAGCACTGCCTTG…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
187,"""FOXA3""","""271G>A""","""Gly91Arg""","""ENSG00000170608""","""19""",45872276,"""NC_000019.10:45872276:G:A""","""ALE00000013""","""CCSBVarC000013""","""393""","""NULL""","""NULL""","""GDEhDisVCh_40008""","""G09""","""RC4""","""ATGCTGGGCTCAGTGAAGATGGAGGCCCAT…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
286,"""ABHD5""","""245A>G""","""His82Arg""","""ENSG00000011198""","""3""",43702326,"""NC_000003.12:43702326:A:G""","""ALE00000031""","""CCSBVarC000031""","""782""","""NULL""","""NULL""","""GDEhDisVCh_40077""","""H08""","""RC4""","""ATGGCGGCGGAGGAGGAGGAGGTGGACTCT…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4765,"""ITGB2""","""769C>T""","""Arg257Trp""","""ENSG00000160255""","""21""",44900448,"""NC_000021.9:44900448:G:A""","""ALE00008408""","""CCSBVarC008422""","""NULL""","""100749""","""NULL""","""GDEhDisVCh_40016""","""C02""","""BRUH""","""ATGCTGGGCCTGCGCCCCCCACTT

In [9]:
gene_info_clinvar.filter(pl.col("symbol")=="GBA").sort(by="clinvar_aa_change").collect()
## Check why missing, it seems like some alleles just weren't collected by ClinVar yet

orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,mutation_id_rc4,mutation_id_cegs2,mutation_id_edgotyping3,entry_plate,entry_well,collection,sequence,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus
i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32
2,"""GBA""","""160G>C""","""Val54Leu""","""ENSG00000177628""","""1""",155240033,"""NC_000001.11:155240033:C:G""","""ALE0000584""","""CCSBVarC000001""","""6""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""F12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""GBA""","""670C>T""","""Leu224Phe""","""ENSG00000177628""","""1""",155238225,"""NC_000001.11:155238225:G:A""","""ALE00000002""","""CCSBVarC000002""","""73""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""C12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""GBA""","""1483G>C""","""Ala495Pro""","""ENSG00000177628""","""1""",155235217,"""NC_000001.11:155235217:C:G""","""ALE00003889""","""CCSBVarC003889""","""NULL""","""213560""","""NULL""","""GDEhDisVCh_40054""","""G07""","""CEGS2""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",38384,"""single nucleotide variant""","""NM_000157.4(GBA1):c.1483G>C (p…",2629,"""GBA1""","""HGNC:4177""","""Conflicting classifications of…",1,"""Aug 20, 2024""",368060,"""-""","""RCV000079341|RCV003129772""","""MedGen:CN169374|MedGen:C366190…","""not specified|not provided""","""germline""","""germline""","""GRCh38""","""NC_000001.11""","""1""",155235217,155235217,"""1q22""","""criteria provided, conflicting…",5,"""-""","""N""","""ClinGen:CA146980,UniProtKB:P04…",2,93450,155235217,"""C""","""G""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV002556287|SCV002756106|SCV0…","""-""","""-""","""1483G>C ""","""Ala495Pro""","""NM_000157.4""",1
2,"""GBA""","""476G>A""","""Arg159Gln""","""ENSG00000177628""","""1""",155238629,"""NC_000001.11:155238629:C:T""","""ALE00003870""","""CCSBVarC003870""","""NULL""","""213515""","""NULL""","""GDEhDisVCh_40054""","""E09""","""CEGS2""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",19330,"""single nucleotide variant""","""NM_000157.4(GBA1):c.476G>A (p.…",2629,"""GBA1""","""HGNC:4177""","""Pathogenic/Likely pathogenic""",1,"""Jun 24, 2024""",79653797,"""-""","""RCV000004518|RCV000004519|RCV0…","""MONDO:MONDO:0009265,MedGen:C19…","""Gaucher disease type I|Gaucher…","""germline""","""germline""","""GRCh38""","""NC_000001.11""","""1""",155238629,155238629,"""1q22""","""criteria provided, multiple su…",8,"""-""","""N""","""OMIM:606463.0004,ClinGen:CA253…",3,4291,155238629,"""C""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV001423035|SCV001425323|SCV0…","""-""","""-""","""476G>A ""","""Arg159Gln""","""NM_000157.4""",2
2,""

### Merge with gnomad

##### Generate all the position coordinates for looking up varchamp-related variants quickly

Switching to pandas for easier processing of small tsv text files 

In [21]:
gene_info_pd = pd.read_csv(f"{meta_inputs}/VarChamp_Consolidated_allele_collection.tsv", sep="\t")
gene_info_pd["chrom"] = "chr" + gene_info_pd["chr"]
gene_info_pd["ref"] = gene_info_pd["nt_change"].apply(lambda x: x.split('>')[0][-1])
gene_info_pd["alt"] = gene_info_pd["nt_change"].apply(lambda x: x.split('>')[-1][0])

##### Sort out all the vcf files and all their position coordinates for looking up varchamp-related variants

In [ ]:
## Generate all the position coordinates for varchamp-related variants for quicker look-up

# VARIANT_LIST = f"../3_outputs/processed_data/gnomad_allele_frequencies/gnomad_variants_list.txt"
# gene_info_pd[["chrom","chr_pos_38","ref","alt"]].to_csv(VARIANT_LIST, sep="\t", header=False, index=False)

In [53]:
# !conda install bcftools -c conda-forge -y ## install bcftools if you didn't

OUTPUT_DIR = f"../3_outputs/processed_data/gnomad_allele_frequencies"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Process chromosomes
for chrom in CHROMOSOMES:
    vcf_file = f"{VCF_DIR}/gnomad.exomes.v4.1.sites.chr{chrom}.vcf.bgz"
    csi_file = f"{vcf_file}.csi"
    output_file = f"{OUTPUT_DIR}/chr{chrom}_allele_frequencies.txt"
    
    if os.path.exists(vcf_file):
        print(f"Processing chromosome {chrom}...")
        
        if not os.path.exists(csi_file):
            print(f"Indexing {vcf_file}")
            run_command(f"bcftools index --threads 4 {vcf_file}")
        
        print(f"Extracting frequencies for {chrom}")
        view_cmd = f"bcftools view -R {VARIANT_LIST} {vcf_file}"
        query_cmd = "bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT\\t%AF\\n'"
        
        with open(output_file, 'w') as f:
            p1 = subprocess.Popen(view_cmd.split(), stdout=subprocess.PIPE)
            p2 = subprocess.Popen(query_cmd.split(), stdin=p1.stdout, stdout=f)
            p1.stdout.close()
            p2.communicate()
            
        print(f"Saved to {output_file}")
    else:
        print(f"Skipping chromosome {chrom} - file not found")

print("Processing completed")

Processing chromosome 1...
Extracting frequencies for 1
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr1_allele_frequencies.txt
Processing chromosome 2...
Extracting frequencies for 2
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr2_allele_frequencies.txt
Processing chromosome 3...
Extracting frequencies for 3
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr3_allele_frequencies.txt
Processing chromosome 4...
Extracting frequencies for 4
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr4_allele_frequencies.txt
Processing chromosome 5...
Extracting frequencies for 5
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr5_allele_frequencies.txt
Processing chromosome 6...
Extracting frequencies for 6
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr6_allele_frequencies.txt
Processing chromosome 7...
Extracting frequencies for 7
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies

### Merge the gnomad AF files from chromosomes

In [23]:
allele_freq_dfs = []
for chrom in CHROMOSOMES:
    try:
        allele_freq = pl.read_csv(f"../3_outputs/processed_data/gnomad_allele_frequencies/chr{chrom}_allele_frequencies.txt",
                                separator="\t", has_header=False, 
                                new_columns=["chr","chr_pos_38","ref","alt","allele_freq"])
        allele_freq = allele_freq.with_columns(
            pl.col("chr").str.replace(r"\'", "").str.replace(r"\'", "").str.replace("chr", "")
        ).drop_nulls()
        # allele_freq = allele_freq.filter(~pl.all_horizontal(pl.all().is_null()))
        # print(allele_freq)
        allele_freq_dfs.append(allele_freq)
    except:
        continue

allele_freq_all = pl.concat(allele_freq_dfs)
allele_freq_gnomad = allele_freq_all.with_columns(
    pl.concat_str([pl.col("chr"),
                   pl.lit('-'),
                   pl.col("chr_pos_38").cast(pl.String),
                   pl.lit('-'),
                   pl.col("ref"),
                   pl.lit('-'),
                   pl.col("alt")
                   ]).alias("varid"),
    pl.col("allele_freq").alias("gnomad_af")
).drop(pl.col("allele_freq")).to_pandas()

allele_freq_gnomad

,chr,chr_pos_38,ref,alt,varid,gnomad_af
0,1,2028259,C,A,1-2028259-C-A,6.850020e-07
1,1,2028259,C,T,1-2028259-C-T,1.409050e-03
2,1,2028260,G,A,1-2028260-G-A,2.174040e-02
3,1,5907193,CG,C,1-5907193-CG-C,0.000000e+00
4,1,5907194,G,A,1-5907194-G-A,1.325610e-05
...,...,...,...,...,...,...
9708,X,154863124,C,T,X-154863124-C-T,1.093600e-05
9709,X,154863125,G,A,X-154863125-G-A,3.645470e-06
9710,X,154863151,C,T,X-154863151-C-T,3.651830e-06
9711,X,155280059,G,C,X-155280059-G-C,1.160730e-04


### Merge the gnomad AF to VarChAMP alleles

In [ ]:
gene_info_pd["varid"] = gene_info_pd["chr"] + "-" + gene_info_pd["chr_pos_38"].astype(str) + "-" + \
                        gene_info_pd["ref"] + "-" + gene_info_pd["alt"]

gene_info_gnomad = gene_info_pd.merge(allele_freq_gnomad,
                                      on="varid",
                                      how="left")
# gene_info_gnomad
# gene_info_gnomad.to_csv("../3_outputs/varchamp_consolidated_alleles_gnomad_annot.tsv", sep="\t", index=False)

In [25]:
gene_info_gnomad

,orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr_x,chr_pos_38_x,spdi,ccsb_allele_id,ccsb_mutation_id,...,sequence,chrom,ref_x,alt_x,varid,chr_y,chr_pos_38_y,ref_y,alt_y,gnomad_af
0,2,GBA,160G>C,Val54Leu,ENSG00000177628,1,155240033,NC_000001.11:155240033:C:G,ALE0000584,CCSBVarC000001,...,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,chr1,G,C,1-155240033-G-C,NaN,NaN,NaN,NaN,NaN
1,2,GBA,670C>T,Leu224Phe,ENSG00000177628,1,155238225,NC_000001.11:155238225:G:A,ALE00000002,CCSBVarC000002,...,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,chr1,C,T,1-155238225-C-T,NaN,NaN,NaN,NaN,NaN
2,2,GBA,887G>A,Arg296Gln,ENSG00000177628,1,155237453,NC_000001.11:155237453:C:T,ALE00000003,CCSBVarC000003,...,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,chr1,G,A,1-155237453-G-A,NaN,NaN,NaN,NaN,NaN
3,2,GBA,1448T>C,Leu483Pro,ENSG00000177628,1,155235252,NC_000001.11:155235252:A:G,ALE00000004,CCSBVarC000004,...,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,chr1,T,C,1-155235252-T-C,NaN,NaN,NaN,NaN,NaN
4,25,SLC9A3R1,328C>G,Leu110Val,ENSG00000109062,17,74749174,NC_000017.11:74749174:C:G,ALE00000005,CCSBVarC000005,...,ATGAGCGCGGACGCAGCGGCCGGGGCGCCCCTGCCCCGGCTCTGCT...,chr17,C,G,17-74749174-C-G,17,74749174.0,C,G,0.026302
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8483,8999,TTR,355G>A,Asp119Asn,ENSG00000118271,18,31598586,NC_000018.10:31598586:G:A,ALE00008468,CCSBVarC008481,...,ATGGCTTCTCATCGTCTGCTCCTCCTCTGCCTTGCTGGACTGGTAT...,chr18,G,A,18-31598586-G-A,18,31598586.0,G,A,0.000079
8484,7323,PMP22,152A>G,His51Arg,ENSG00000109099,17,15259120,NC_000017.11:15259120:T:C,ALE00008469,CCSBVarC008466,...,ATGCTCCTCCTGTTGCTGAGTATCATCGTCCTCCACGTCGCGGTGC...,chr17,A,G,17-15259120-A-G,NaN,NaN,NaN,NaN,NaN
8485,2651,CLN6,89G>C,Gly30Ala,ENSG00000128973,15,68218645,NC_000015.10:68218645:C:G,ALE00008470,CCSBVarC008436,...,ATGGAGGCGACGCGGAGGCGGCAGCACCTGGGAGCGACGGGCGGCC...,chr15,G,C,15-68218645-G-C,NaN,NaN,NaN,NaN,NaN
8486,7323,PMP22,403G>A,Ala135Thr,ENSG00000109099,17,15230997,NC_000017.11:15230997:C:T,ALE00008471,CCSBVarC008467,...,ATGCTCCTCCTGTTGCTGAGTATCATCGTCCTCCACGTCGCGGTGC...,chr17,G,A,17-15230997-G-A,NaN,NaN,NaN,NaN,NaN


### Merge the ClinVar info AND gnomad AF BOTH to VarChAMP alleles

In [33]:
gene_info_clinvar_pd = gene_info_clinvar.collect().to_pandas()
gene_info_clinvar_pd["chrom"] = "chr" + gene_info_clinvar_pd["chr"]
gene_info_clinvar_pd["ref"] = gene_info_clinvar_pd["nt_change"].apply(lambda x: x.split('>')[0][-1])
gene_info_clinvar_pd["alt"] = gene_info_clinvar_pd["nt_change"].apply(lambda x: x.split('>')[-1][0])
gene_info_clinvar_pd["varid"] = gene_info_clinvar_pd["chr"] + "-" + gene_info_clinvar_pd["chr_pos_38"].astype(str) + "-" + \
                                gene_info_clinvar_pd["ref"] + "-" + gene_info_clinvar_pd["alt"]
gene_info_clinvar_gnomad_pd = gene_info_clinvar_pd.merge(allele_freq_gnomad[["varid","gnomad_af"]],
                                                  on="varid",how="left")
gene_info_clinvar_gnomad_pd.to_csv("../3_outputs/varchamp_consolidated_alleles_clinvar_gnomad.tsv", sep="\t", index=False)

In [34]:
# gene_info_clinvar_gnomad_pd
gene_info_clinvar_gnomad_pd[gene_info_clinvar_gnomad_pd["gnomad_af"].isna()]

,orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,...,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,chrom,ref,alt,varid,gnomad_af
0,2,GBA,160G>C,Val54Leu,ENSG00000177628,1,155240033,NC_000001.11:155240033:C:G,ALE0000584,CCSBVarC000001,...,None,None,None,None,NaN,chr1,G,C,1-155240033-G-C,NaN
1,2,GBA,670C>T,Leu224Phe,ENSG00000177628,1,155238225,NC_000001.11:155238225:G:A,ALE00000002,CCSBVarC000002,...,None,None,None,None,NaN,chr1,C,T,1-155238225-C-T,NaN
2,2,GBA,887G>A,Arg296Gln,ENSG00000177628,1,155237453,NC_000001.11:155237453:C:T,ALE00000003,CCSBVarC000003,...,-,887G>A,Arg296Gln,NM_000157.4,2.0,chr1,G,A,1-155237453-G-A,NaN
3,2,GBA,1448T>C,Leu483Pro,ENSG00000177628,1,155235252,NC_000001.11:155235252:A:G,ALE00000004,CCSBVarC000004,...,-,1448T>C,Leu483Pro,NM_000157.4,2.0,chr1,T,C,1-155235252-T-C,NaN
6,138,MYD88,278T>C,Leu93Pro,ENSG00000172936,3,38138978,NC_000003.12:38138978:T:C,ALE00000007,CCSBVarC000007,...,-,278T>C,Leu93Pro,NM_002468.5,NaN,chr3,T,C,3-38138978-T-C,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8478,6666,MAP2K2,49A>G,Thr17Ala,ENSG00000126934,19,4123827,NC_000019.10:4123827:T:C,ALE00008463,CCSBVarC008461,...,-,49A>G,Thr17Ala,NM_030662.4,1.0,chr19,A,G,19-4123827-A-G,NaN
8484,7323,PMP22,152A>G,His51Arg,ENSG00000109099,17,15259120,NC_000017.11:15259120:T:C,ALE00008469,CCSBVarC008466,...,-,152A>G,His51Arg,NM_000304.4,1.0,chr17,A,G,17-15259120-A-G,NaN
8485,2651,CLN6,89G>C,Gly30Ala,ENSG00000128973,15,68218645,NC_000015.10:68218645:C:G,ALE00008470,CCSBVarC008436,...,-,89G>C,Gly30Ala,NM_017882.3,1.0,chr15,G,C,15-68218645-G-C,NaN
8486,7323,PMP22,403G>A,Ala135Thr,ENSG00000109099,17,15230997,NC_000017.11:15230997:C:T,ALE00008471,CCSBVarC008467,...,-,403G>A,Ala135Thr,NM_000304.4,2.0,chr17,G,A,17-15230997-G-A,NaN
